# MinerU LayoutExtractor Test Notebook

This notebook demonstrates and tests the **MinerUExtractor** strategy (an alternative Strategy B)
which uses **MinerU** for layout-aware extraction via a normalised JSON schema:

- Table-heavy and mixed-layout documents
- OCR-heavy or scanned PDFs (when MinerU has been run with OCR)
- Normalisation into the shared `ExtractedDocument` schema via `MinerUAdapter`.

> Prerequisites:
> - MinerU installed, e.g. `uv pip install -U "mineru[all]"`
> - MinerU has already been run on the chosen PDF, producing a JSON file
>   (e.g. `<pdf>.mineru.json`) matching the schema expected by `MinerUAdapter`.


In [ ]:
# Setup and Imports

import sys
from pathlib import Path

# Ensure src is on the path
sys.path.insert(0, str(Path().absolute().parent))

from src.agents.triage import TriageAgent
from src.strategies import MinerUExtractor
from src.models.document_profile import DocumentProfile


In [ ]:
# Select Test Document and MinerU JSON

DATA_ROOT = Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data")

TEST_DOCS = {
    "class_a": DATA_ROOT / "class_a" / "CBE_ANNUAL_REPORT_2023-24.pdf",
    "class_b": DATA_ROOT / "class_b" / "2018_Audited_Financial_Statement_Report.pdf",
    "class_c": DATA_ROOT / "class_c" / "fta_performance_survey_final_report_2022.pdf",
    "class_d": DATA_ROOT / "class_d" / "tax_expenditure_ethiopia_2021_22.pdf",
}

selected_doc = "class_b"  # change as needed
pdf_path = TEST_DOCS[selected_doc]

# Default MinerU JSON path convention: <pdf>.mineru.json
mineru_json_path = pdf_path.with_suffix(pdf_path.suffix + ".mineru.json")

print(f"PDF: {pdf_path}")
print(f"Expected MinerU JSON: {mineru_json_path}")
print(f"PDF exists: {pdf_path.exists()}")
print(f"MinerU JSON exists: {mineru_json_path.exists()}")


In [ ]:
# Step 1: Triage and Profile

profiles_dir = Path(".refinery/profiles")
profiles_dir.mkdir(parents=True, exist_ok=True)
triage = TriageAgent(profiles_dir=profiles_dir)

print("🔍 Running Triage Agent...")
profile: DocumentProfile = triage.classify_document(pdf_path)

print("\n📋 Document Profile (summary):")
print(f"  - Document ID: {profile.doc_id}")
print(f"  - Origin: {profile.origin_type}")
print(f"  - Layout: {profile.layout_complexity}")
print(f"  - Domain: {profile.domain_hint}")
print(f"  - Estimated Cost: {profile.estimated_cost}")
print(f"  - Language: {profile.language} (confidence: {profile.language_confidence:.2f})")
print(f"  - Pages: {profile.metadata.page_count}")


In [1]:
# Step 1.5: Generate MinerU JSON if Missing

# Check if MinerU JSON exists, and if not, try to generate it
if not mineru_json_path.exists():
    print("⚠️  MinerU JSON not found. Attempting to generate it...")
    print(f"   Expected path: {mineru_json_path}\n")
    
    import subprocess
    import time
    import json
    import shutil
    import sys
    
    # Strategy 1: Try Python API first
    python_api_success = False
    try:
        from mineru import MinerU
        print("✓ MinerU Python API available")
        print(f"📄 Running MinerU on: {pdf_path.name}")
        print("   This may take several minutes for large documents...\n")
        
        start_time = time.time()
        mineru_instance = MinerU()
        
        # Try to extract
        result = mineru_instance.extract(str(pdf_path))
        
        # Handle different return types
        if isinstance(result, (str, Path)):
            output_path = Path(result)
            if output_path.exists() and output_path.suffix == '.json':
                shutil.copy2(str(output_path), str(mineru_json_path))
                python_api_success = True
        elif isinstance(result, dict):
            with open(mineru_json_path, 'w', encoding='utf-8') as f:
                json.dump(result, f, indent=2, ensure_ascii=False)
            python_api_success = True
        
        if python_api_success:
            elapsed = time.time() - start_time
            print(f"✓ MinerU extraction completed in {elapsed:.1f} seconds\n")
            
    except ImportError as e:
        print(f"⚠️  MinerU Python API not available: {e}")
        print("   Falling back to CLI...\n")
    except Exception as e:
        print(f"⚠️  MinerU Python API failed: {e}")
        print("   Falling back to CLI...\n")
    
    # Strategy 2: Try CLI if Python API didn't work
    if not python_api_success and not mineru_json_path.exists():
        print("📄 Attempting MinerU via CLI...")
        print(f"   Processing: {pdf_path.name}")
        print("   This may take several minutes for large documents...\n")
        
        output_dir = mineru_json_path.parent
        start_time = time.time()
        
        # Try different CLI command patterns
        # Based on error: MinerU requires -p/--path flag
        cli_patterns = [
            # Pattern 1: mineru -p <input> -o <output>
            ["mineru", "-p", str(pdf_path), "-o", str(output_dir)],
            # Pattern 2: mineru --path <input> --output <output>
            ["mineru", "--path", str(pdf_path), "--output", str(output_dir)],
            # Pattern 3: python -m mineru -p <input> -o <output>
            ["python", "-m", "mineru", "-p", str(pdf_path), "-o", str(output_dir)],
            # Pattern 4: Try with venv python if available
            [sys.executable, "-m", "mineru", "-p", str(pdf_path), "-o", str(output_dir)],
        ]
        
        cli_success = False
        for cmd in cli_patterns:
            try:
                print(f"   Trying: {' '.join(cmd)}")
                result = subprocess.run(
                    cmd, 
                    capture_output=True, 
                    text=True, 
                    timeout=1800,  # 30 minutes for large docs
                    cwd=str(output_dir)
                )
                
                if result.returncode == 0:
                    # Look for generated JSON file in various locations
                    # MinerU may create subdirectories or use different naming
                    possible_locations = [
                        # Direct matches
                        mineru_json_path,
                        output_dir / f"{pdf_path.stem}.json",
                        output_dir / f"{pdf_path.stem}.mineru.json",
                        output_dir / f"{pdf_path.name}.json",
                        output_dir / f"{pdf_path.name}.mineru.json",
                        # In subdirectories (MinerU often creates output folders)
                        output_dir / pdf_path.stem / f"{pdf_path.stem}.json",
                        output_dir / pdf_path.stem / "output.json",
                        output_dir / "output" / f"{pdf_path.stem}.json",
                    ]
                    
                    # Also search recursively in output_dir for any .json files
                    json_files_found = list(output_dir.rglob("*.json"))
                    
                    for possible in possible_locations + json_files_found:
                        if possible.exists() and possible.is_file():
                            if possible != mineru_json_path:
                                # Ensure parent directory exists
                                mineru_json_path.parent.mkdir(parents=True, exist_ok=True)
                                shutil.copy2(str(possible), str(mineru_json_path))
                                print(f"   Found JSON at: {possible}")
                            cli_success = True
                            elapsed = time.time() - start_time
                            print(f"✓ MinerU JSON generated in {elapsed:.1f} seconds: {mineru_json_path}\n")
                            break
                    
                    if cli_success:
                        break
                else:
                    print(f"   Command failed (exit code {result.returncode})")
                    if result.stderr:
                        print(f"   Error: {result.stderr[:200]}")
                    
            except subprocess.TimeoutExpired:
                print(f"   Command timed out after 30 minutes")
                continue
            except FileNotFoundError:
                print(f"   Command not found, trying next pattern...")
                continue
            except Exception as e:
                print(f"   Error: {e}")
                continue
        
        if not cli_success:
            print("\n❌ Could not generate MinerU JSON automatically.\n")
            print("📝 Please run MinerU manually using one of these methods:\n")
            print("   Method 1 (CLI - correct syntax):")
            print(f"   mineru -p \"{pdf_path}\" -o \"{output_dir}\"")
            print(f"   # or")
            print(f"   mineru --path \"{pdf_path}\" --output \"{output_dir}\"\n")
            print("   Method 2 (Python - check MinerU docs for correct API):")
            print("   from mineru import MinerU  # or check actual import path")
            print("   mu = MinerU()")
            print(f"   mu.extract('{pdf_path}')\n")
            print("   Method 3 (Use helper script):")
            print(f"   python scripts/generate_mineru_json.py \"{pdf_path}\"\n")
            print("   Method 4 (Check MinerU help for exact syntax):")
            print("   mineru --help")
            print("   Visit: https://github.com/opendatalab/MinerU\n")
            print(f"   Then ensure the output JSON is saved/renamed to:\n   {mineru_json_path}\n")
            print("   Note: MinerU may output with a different filename. Check the output directory.")
            raise FileNotFoundError(
                f"MinerU JSON not found and could not be auto-generated.\n"
                f"Please run MinerU manually and ensure the output is saved to:\n{mineru_json_path}"
            )
else:
    print(f"✓ MinerU JSON found: {mineru_json_path}")

NameError: name 'mineru_json_path' is not defined

In [ ]:
# Step 2a: Generate MinerU Markdown Output

# MinerU can output Markdown directly. Let's try to get it.
print("📝 Attempting to get MinerU Markdown output...")

import subprocess
import time
import json

markdown_path = pdf_path.with_suffix('.md')
markdown_content = None

# Try to get Markdown from MinerU CLI
# MinerU typically outputs to a subdirectory with markdown files
output_dir = mineru_json_path.parent

# Check if MinerU has already been run and markdown exists
possible_md_paths = [
    output_dir / f"{pdf_path.stem}.md",
    output_dir / pdf_path.stem / f"{pdf_path.stem}.md",
    output_dir / pdf_path.stem / "output.md",
    output_dir / "output" / f"{pdf_path.stem}.md",
]

# Also search recursively
md_files_found = list(output_dir.rglob("*.md"))

for md_path in possible_md_paths + md_files_found:
    if md_path.exists() and md_path.is_file():
        print(f"✓ Found existing Markdown: {md_path}")
        with open(md_path, 'r', encoding='utf-8') as f:
            markdown_content = f.read()
        # Copy to expected location
        import shutil
        shutil.copy2(str(md_path), str(markdown_path))
        print(f"✓ Copied to: {markdown_path}")
        break

# If not found, try to generate it with MinerU
if markdown_content is None:
    print("⚠️  Markdown not found. Attempting to generate with MinerU...")
    print("   (This may take several minutes...)")
    
    start_time = time.time()
    # Try MinerU with markdown output
    cmd = [sys.executable, "-m", "mineru", "-p", str(pdf_path), "-o", str(output_dir)]
    
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=1800,
            cwd=str(output_dir)
        )
        
        if result.returncode == 0:
            # Look for generated markdown
            for md_path in possible_md_paths + list(output_dir.rglob("*.md")):
                if md_path.exists() and md_path.is_file():
                    with open(md_path, 'r', encoding='utf-8') as f:
                        markdown_content = f.read()
                    import shutil
                    shutil.copy2(str(md_path), str(markdown_path))
                    elapsed = time.time() - start_time
                    print(f"✓ Markdown generated in {elapsed:.1f} seconds: {markdown_path}")
                    break
        else:
            print(f"⚠️  MinerU CLI failed. You may need to run manually:")
            print(f"   mineru -p \"{pdf_path}\" -o \"{output_dir}\"")
            print(f"   Then look for .md files in: {output_dir}")
    except Exception as e:
        print(f"⚠️  Could not generate Markdown automatically: {e}")
        print(f"   You can run MinerU manually to get Markdown output")

if markdown_content:
    print(f"\n✓ Markdown available: {markdown_path}")
    print(f"  - File size: {len(markdown_content):,} characters")
    print(f"  - Lines: {len(markdown_content.splitlines()):,}")
    
    # Show preview
    print("\n📄 Markdown Preview (first 500 characters):")
    print("=" * 80)
    print(markdown_content[:500])
    print("...")
else:
    print("\n⚠️  Markdown not available. Proceeding with JSON extraction only.")


In [ ]:
# Step 2b: Analyze MinerU Markdown Content

if markdown_content:
    import re
    from collections import Counter
    
    print("🔍 Analyzing MinerU Markdown structure...\n")
    
    # Count different Markdown elements
    analysis = {
        "headings": {
            "h1": len(re.findall(r'^#\s+', markdown_content, re.MULTILINE)),
            "h2": len(re.findall(r'^##\s+', markdown_content, re.MULTILINE)),
            "h3": len(re.findall(r'^###\s+', markdown_content, re.MULTILINE)),
            "h4": len(re.findall(r'^####\s+', markdown_content, re.MULTILINE)),
        },
        "tables": len(re.findall(r'^\|.*\|', markdown_content, re.MULTILINE)) // 3,
        "code_blocks": len(re.findall(r'```', markdown_content)) // 2,
        "links": len(re.findall(r'\[.*?\]\(.*?\)', markdown_content)),
        "images": len(re.findall(r'!\[.*?\]\(.*?\)', markdown_content)),
        "lists": {
            "unordered": len(re.findall(r'^[\*\-\+]\s+', markdown_content, re.MULTILINE)),
            "ordered": len(re.findall(r'^\d+\.\s+', markdown_content, re.MULTILINE)),
        },
        "formulas": len(re.findall(r'\$.*?\$', markdown_content)) + len(re.findall(r'\$\$.*?\$\$', markdown_content, re.DOTALL)),
    }
    
    print("📊 MinerU Markdown Structure Analysis:")
    print("=" * 80)
    print(f"Headings:")
    print(f"  - H1: {analysis['headings']['h1']}")
    print(f"  - H2: {analysis['headings']['h2']}")
    print(f"  - H3: {analysis['headings']['h3']}")
    print(f"  - H4: {analysis['headings']['h4']}")
    print(f"\nTables: ~{analysis['tables']}")
    print(f"Code blocks: {analysis['code_blocks']}")
    print(f"Links: {analysis['links']}")
    print(f"Images: {analysis['images']}")
    print(f"Formulas (LaTeX): {analysis['formulas']}")
    print(f"Lists:")
    print(f"  - Unordered: {analysis['lists']['unordered']}")
    print(f"  - Ordered: {analysis['lists']['ordered']}")
    
    # Extract heading structure
    headings = re.findall(r'^(#{1,4})\s+(.+)$', markdown_content, re.MULTILINE)
    if headings:
        print(f"\n📑 Document Outline (first 10 headings):")
        print("=" * 80)
        for level, title in headings[:10]:
            indent = "  " * (len(level) - 1)
            print(f"{indent}{level} {title}")
    
    # Find formulas (MinerU converts formulas to LaTeX)
    formulas = re.findall(r'\$\$([^\$]+)\$\$', markdown_content, re.DOTALL)
    if formulas:
        print(f"\n🔢 Sample Formulas (LaTeX, first 3):")
        print("=" * 80)
        for i, formula in enumerate(formulas[:3], 1):
            print(f"{i}. {formula.strip()[:100]}...")
else:
    print("⚠️  Skipping Markdown analysis (Markdown not available)")

In [ ]:
# Step 2c: Convert MinerU JSON to ExtractedDocument (Optional)

# Final check - JSON should exist after Step 1.5
if not mineru_json_path.exists():
    raise FileNotFoundError(
        f"MinerU JSON not found at {mineru_json_path}.\n"
        "Please run Step 1.5 to generate it, or run MinerU manually:\n"
        f"  mineru -p \"{pdf_path}\" -o \"{pdf_path.parent}\""
    )

print(f"✓ Using MinerU JSON: {mineru_json_path}")
mineru_extractor = MinerUExtractor(mineru_json_path=mineru_json_path)

print("\n📄 Converting MinerU JSON → ExtractedDocument...")
print("   (For structured analysis after Markdown investigation)")

import time
start_time = time.time()

extracted = mineru_extractor.extract(str(pdf_path))

elapsed = time.time() - start_time
print(f"\n✓ Extraction completed in {elapsed:.2f} seconds")
print("\n📊 Extraction Summary (MinerU → ExtractedDocument):")
print(f"  - Text Blocks: {len(extracted.text_blocks):,}")
print(f"  - Tables: {len(extracted.tables)}")
print(f"  - Figures: {len(extracted.figures)}")
print(f"  - Reading Order indices: {len(extracted.reading_order)}")

In [ ]:
# Step 3: Inspect Sample Tables and Figures

print("📊 Sample Tables (first 3):")
print("=" * 80)
for i, table in enumerate(extracted.tables[:3], start=1):
    print(f"\n[Table {i}] Page {table.page_num}")
    print(f"  Headers: {table.headers}")
    if table.rows:
        print(f"  First row: {table.rows[0]}")

print("\n🖼️ Sample Figures (first 3):")
print("=" * 80)
for i, fig in enumerate(extracted.figures[:3], start=1):
    print(f"\n[Figure {i}] Page {fig.page_num}")
    print(f"  Caption: {fig.caption or '(none)'}")
    print(f"  BBox: ({fig.bbox.x0:.1f}, {fig.bbox.y0:.1f}) → ({fig.bbox.x1:.1f}, {fig.bbox.y1:.1f})")
